In [ ]:
# ===== Cell 1: setup + TRTR anchor (same pipeline as Oct 14, + BA) =====
import pandas as pd, numpy as np, glob, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

SEED = 42
ID_COLS = ['flow_key', 'timeframe_id', 'ip_a', 'ip_b', 'port_a', 'port_b']
CONTEXT_PREFIX = ('target_', 'source_', 'flow_share_')        # 21 window-level cols -> removed for "hard"

def to_label(s):
    return s.astype(str).str.strip().str.lower().map({'true':1,'false':0,'1':1,'0':0,'1.0':1,'0.0':0})

train = pd.read_csv('/content/train.csv'); test = pd.read_csv('/content/test.csv')
train['is_attack'] = to_label(train['is_attack']); test['is_attack'] = to_label(test['is_attack'])
benign_train = train[train.is_attack == 0].reset_index(drop=True)
attack_train = train[train.is_attack == 1].reset_index(drop=True)
y_test = test['is_attack'].astype(int).values
print('train', train.shape, '| attack', len(attack_train), 'benign', len(benign_train),
      '| test attack', int(y_test.sum()), 'benign', int((y_test == 0).sum()))

def load_syn_df(df):
    s = df.copy()
    for c in train.columns:
        if c not in s.columns: s[c] = np.nan
    s = s[train.columns]
    for c in s.columns:
        if c in ID_COLS[:1] + ['ip_a', 'ip_b']: continue          # text ids
        if c == 'is_attack': s[c] = 1
        else: s[c] = pd.to_numeric(s[c], errors='coerce')
    return s

def prep_full(df):
    X = df.drop(columns=[c for c in ID_COLS + ['is_attack'] if c in df.columns]).copy()
    for c in X.select_dtypes('bool').columns: X[c] = X[c].astype(int)
    return X

def prep_hard(df):
    X = prep_full(df)
    return X.drop(columns=[c for c in X.columns if c.startswith(CONTEXT_PREFIX)])

def make_models():
    return {'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=SEED, n_jobs=-1),
            'LogisticReg':  LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED),
            'SVM':          LinearSVC(C=1.0, class_weight='balanced', max_iter=5000, random_state=SEED)}
MODELS = list(make_models().keys())

def fit_predict(attack_rows, prep_fn=prep_hard):
    a = attack_rows.copy(); a['is_attack'] = 1
    tr = pd.concat([a, benign_train], ignore_index=True)
    Xtr, ytr = prep_fn(tr), tr['is_attack'].astype(int)
    Xte = prep_fn(test)[Xtr.columns]
    preds = {}
    for name, m in make_models().items():
        pipe = make_pipeline(SimpleImputer(strategy='constant', fill_value=0, keep_empty_features=True),
                             StandardScaler(), m).fit(Xtr, ytr)
        preds[name] = pipe.predict(Xte).astype(int)
    return preds

def score(yp, generator, model):
    tn, fp, fn, tp = confusion_matrix(y_test, yp).ravel()
    rec, tnr = tp / (tp + fn), tn / (tn + fp)
    return {'Generator': generator, 'Model': model, 'Acc': accuracy_score(y_test, yp),
            'Prec': precision_score(y_test, yp, zero_division=0), 'Rec': rec, 'F1': f1_score(y_test, yp),
            'Benign_FPR': fp / (fp + tn), 'BA': (rec + tnr) / 2, 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp}

# ---------- TRTR anchor ----------
print('hard features:', prep_hard(test).shape[1])
PREDS = {'Real (TRTR)': fit_predict(attack_train)}
res = pd.DataFrame([score(PREDS['Real (TRTR)'][m], 'Real (TRTR)', m) for m in MODELS])
print(res[['Model', 'F1', 'BA', 'TN', 'FP', 'FN', 'TP']].round(3).to_string(index=False))

# must match the saved Oct results exactly (tstr_hard_sample.csv)
EXPECT = {'RandomForest': (192, 2, 11, 1335), 'LogisticReg': (192, 2, 15, 1331), 'SVM': (191, 3, 16, 1330)}
ok = all(tuple(res.loc[res.Model == m, ['TN', 'FP', 'FN', 'TP']].iloc[0]) == v for m, v in EXPECT.items())
print('\nTRTR anchor matches Table I counts exactly:', ok)

In [ ]:
# ===== Cell 2: load all synthetic sets + combined (run-1 + run-2) =====
LLM_NAMES = ['Llama', 'Falcon', 'Qwen']
SETTINGS = {'A': '{l}_run{r}.csv',          # benign stats
            'B': '{l}_B_run{r}.csv',        # attack stats
            'C': '{l}_S_run{r}.csv'}        # attack stats + 10 real flows
RUNS = (1, 2)

syn_raw = {}
for st, pat in SETTINGS.items():
    for g in LLM_NAMES:
        for r in RUNS:
            syn_raw[f'{g}_{st}_run{r}'] = pd.read_csv('/content/' + pat.format(l=g.lower(), r=r))
for g in LLM_NAMES:
    syn_raw[f'{g}_desc'] = pd.read_csv(f'/content/desc_only_{g.lower()}.csv')
syn_raw['Heuristic'] = pd.read_csv('/content/heuristic.csv')

SYN = {k: load_syn_df(v) for k, v in syn_raw.items()}
for st in SETTINGS:
    for g in LLM_NAMES:
        SYN[f'{g}_{st}_combined'] = pd.concat([SYN[f'{g}_{st}_run{r}'] for r in RUNS], ignore_index=True)

# ---------- sanity ----------
print(f'{"set":20s} {"rows":>5s} {"cols":>4s} {"dups":>5s}  missing_cols')
for k, df in syn_raw.items():
    miss = [c for c in train.columns if c not in df.columns]
    print(f'{k:20s} {len(df):5d} {df.shape[1]:4d} {df.duplicated().sum():5d}  {miss[:3] if miss else "-"}')
print('\ncombined sets:', {k: len(v) for k, v in SYN.items() if k.endswith('_combined')})

In [ ]:
# ===== Cell 3: TSTR (RF/LR/SVM) on all sets + F1 check vs paper =====
for g, s in SYN.items():
    print('training', g); PREDS[g] = fit_predict(s)

res = pd.DataFrame([score(PREDS[g][m], g, m) for g in PREDS for m in MODELS])
res.to_csv('tstr_hard_all_BA.csv', index=False)

F1 = res.pivot(index='Generator', columns='Model', values='F1')[MODELS]
BA = res.pivot(index='Generator', columns='Model', values='BA')[MODELS]
F1['Mean'] = F1[MODELS].mean(axis=1); BA['Mean'] = BA[MODELS].mean(axis=1)

# ---------- expected F1 (RF, LR, SVM) from Table I / Table III / tstr_hard_sample ----------
EXP = {
 'Heuristic': (.404, .910, .087),
 'Llama_A_run1': (.906, .713, .813), 'Llama_A_run2': (.906, .644, .716), 'Llama_A_combined': (.906, .705, .761),
 'Falcon_A_run1': (.910, .908, .907), 'Falcon_A_run2': (.909, .907, .906), 'Falcon_A_combined': (.911, .907, .906),
 'Qwen_A_run1': (.907, .889, .903), 'Qwen_A_run2': (.911, .897, .904), 'Qwen_A_combined': (.911, .897, .904),
 'Llama_B_run1': (.904, .894, .892), 'Llama_B_run2': (.904, .895, .891), 'Llama_B_combined': (.943, .895, .890),
 'Falcon_B_run1': (.994, .861, .871), 'Falcon_B_run2': (.992, .896, .892), 'Falcon_B_combined': (.994, .891, .880),
 'Qwen_B_run1': (.907, .906, .903), 'Qwen_B_run2': (.907, .905, .904), 'Qwen_B_combined': (.908, .907, .904),
 'Llama_desc': (.902, .000, .013), 'Falcon_desc': (.909, .704, .895), 'Qwen_desc': (.906, .894, .901),
 'Llama_C_run1': (.993, .985, .982), 'Falcon_C_run1': (.989, .986, .983), 'Qwen_C_run1': (.995, .986, .930),
}
bad = {}
for g, v in EXP.items():
    got = tuple(F1.loc[g, MODELS].round(3))
    if any(abs(a - b) > .0005 for a, b in zip(got, v)): bad[g] = {'paper': v, 'now': got}
print('\nF1 matches paper for all', len(EXP), 'checked sets:', not bad)
for g, d in bad.items(): print(f'  MISMATCH {g}: paper {d["paper"]} | now {d["now"]}')

# ---------- quick look ----------
out = pd.concat({'F1': F1, 'BA': BA}, axis=1).round(3)
print('\n', out.to_string())

In [ ]:
##### now ctgan sandharva
!pip install -q "sdv<2"


In [ ]:
# ===== Cell 4b: CTGAN k=2,000 retrain (Oct 13 setup) + TSTR F1/BA + fidelity =====
import os, time, random, gc, shutil
import numpy as np, pandas as pd, torch
from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer
from rdt.transformers.numerical import FloatFormatter
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics.pairwise import euclidean_distances
from scipy.stats import ks_2samp
from scipy.spatial.distance import jensenshannon

N_SYN, EPOCHS, SEEDS, K, SUB_SEED = 3300, 300, [42, 45, 50], 2000, 42
KDIR = '/content/ctgan_k2000'; os.makedirs(KDIR, exist_ok=True)
KEY_PARTS = ['timeframe_id', 'proto', 'ip_a', 'ip_b', 'port_a', 'port_b']
print('GPU available:', torch.cuda.is_available())

# ---------- same inputs as Oct 13 (metadata built from all 3,116 train attacks) ----------
atk = attack_train.drop(columns=['is_attack']).reset_index(drop=True)
all_nan     = [c for c in atk.columns if atk[c].isna().all()]
constant    = [c for c in atk.columns if c not in all_nan and atk[c].nunique(dropna=False) == 1]
partial_nan = [c for c in atk.columns if c not in all_nan and atk[c].isna().any()]
gan_df = atk.drop(columns=['flow_key'] + all_nan)
orig_dtypes = atk.dtypes
cols = {c: ({'sdtype': 'categorical'} if gan_df[c].dtype == object or gan_df[c].nunique(dropna=True) <= 10
            else {'sdtype': 'numerical'}) for c in gan_df.columns}
metadata = Metadata.load_from_dict({'tables': {'attack': {'columns': cols}}})
num_nan_cols = [c for c in partial_nan if c in gan_df.columns and cols[c]['sdtype'] == 'numerical']
n_cat = sum(v['sdtype'] == 'categorical' for v in cols.values())
print(f'all-NaN {len(all_nan)} | constant {len(constant)} | CTGAN models {gan_df.shape[1]} cols '
      f'({n_cat} categorical, {gan_df.shape[1]-n_cat} numerical) | NaN-pattern cols {len(num_nan_cols)}')

# ---------- same nested, timeframe-stratified subset (subset seed 42) ----------
rng = np.random.default_rng(SUB_SEED)
pos = pd.Series(np.nan, index=atk.index)
for tf, idx in atk.groupby('timeframe_id').groups.items():
    idx = rng.permutation(np.array(idx))
    pos.loc[idx] = (np.arange(len(idx)) + rng.random(len(idx))) / len(idx)
sub = atk.loc[pos.sort_values().index[:K]].reset_index(drop=True)
sub.assign(is_attack=True).to_csv(f'{KDIR}/train_attack_k{K}.csv', index=False)
print('k=2000 timeframe share:', sub['timeframe_id'].value_counts(normalize=True).sort_index().round(3).to_dict())
print('Oct 13 share         : {5: 0.134, 6: 0.177, 7: 0.14, 8: 0.201, 9: 0.17, 10: 0.118, 11: 0.06}')

# ---------- same CTGAN as Oct 13 ----------
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def train_ctgan_k(sub, seed, n=N_SYN):
    set_seed(seed)
    df_k = sub.drop(columns=['flow_key'] + all_nan).copy()
    synth = CTGANSynthesizer(metadata, enforce_min_max_values=True, enforce_rounding=True,
                             epochs=EPOCHS, batch_size=500, pac=10, embedding_dim=128,
                             generator_dim=(256, 256), discriminator_dim=(256, 256),
                             generator_lr=2e-4, discriminator_lr=2e-4,
                             generator_decay=1e-6, discriminator_decay=1e-6,
                             discriminator_steps=1, log_frequency=True, verbose=False,
                             enable_gpu=torch.cuda.is_available())
    synth.auto_assign_transformers(df_k)
    if num_nan_cols:
        synth.update_transformers({c: FloatFormatter(missing_value_generation='from_column',
                                   learn_rounding_scheme=True, enforce_min_max_values=True) for c in num_nan_cols})
    t0 = time.time(); synth.fit(df_k); t = time.time() - t0
    syn = synth.sample(num_rows=n)
    for c in all_nan: syn[c] = np.nan
    for c in orig_dtypes.index:
        if c in syn.columns and orig_dtypes[c].kind in 'iu' and syn[c].notna().all():
            syn[c] = syn[c].round().astype(orig_dtypes[c])
    syn['flow_key'] = syn[KEY_PARTS].astype(str).agg('|'.join, axis=1)
    syn['is_attack'] = True
    try: synth.get_loss_values().to_csv(f'{KDIR}/loss_k{K}_e{EPOCHS}_s{seed}.csv', index=False)
    except Exception: pass
    return syn[train.columns], t

# ---------- fidelity (same as Oct 13, vs 1,346 real test attacks) ----------
test_att = test[test['is_attack'] == 1].drop(columns=['is_attack'])
R  = prep_hard(test_att).fillna(0).astype(float)
qt = QuantileTransformer(n_quantiles=1000, output_distribution='uniform', random_state=SEED).fit(R)
Rz = qt.transform(R)
def subs(A, n, s): return A[np.random.default_rng(s).choice(len(A), min(n, len(A)), replace=False)]
d0 = euclidean_distances(subs(Rz, 1000, 0), squared=True)
gams = [1.0 / (np.median(d0[d0 > 0]) * f) for f in (0.1, 1, 10)]
Kf = lambda P, Q: sum(np.exp(-g * euclidean_distances(P, Q, squared=True)) for g in gams) / len(gams)
def fidelity(df):
    X = prep_hard(df)[R.columns].fillna(0).astype(float); ks, js = [], []
    for c in R.columns:
        r, x = R[c].values, X[c].values
        if np.ptp(np.r_[r, x]) == 0: ks.append(0.0); js.append(0.0); continue
        ks.append(ks_2samp(r, x).statistic)
        e = np.unique(np.quantile(np.r_[r, x], np.linspace(0, 1, 21)))
        if len(e) < 2: e = np.array([e[0] - .5, e[0] + .5])
        pr, _ = np.histogram(r, e); px, _ = np.histogram(x, e)
        js.append(jensenshannon(pr + 1e-12, px + 1e-12, base=2) ** 2)
    A, B = subs(Rz, 1000, 1), subs(qt.transform(X), 1000, 2); m, k = len(A), len(B)
    Kxx, Kyy, Kxy = Kf(A, A), Kf(B, B), Kf(A, B)
    return {'KS': np.mean(ks), 'JSD': np.mean(js),
            'MMD2': (Kxx.sum()-np.trace(Kxx))/(m*(m-1)) + (Kyy.sum()-np.trace(Kyy))/(k*(k-1)) - 2*Kxy.mean()}

# ---------- train 3 seeds, save, evaluate ----------
crow, logs = [], []
for s in SEEDS:
    path = f'{KDIR}/ctgan_k{K}_e{EPOCHS}_s{s}.csv'
    print(f'[run] seed {s} ...', end=' ', flush=True)
    syn, t = train_ctgan_k(sub, s); syn.to_csv(path, index=False)
    print(f'done in {t/60:.1f} min')
    const_ok = sum((syn[c] == atk[c].iloc[0]).all() for c in constant)
    logs.append({'seed': s, 'rows': len(syn), 'train_min': round(t/60, 2),
                 'dups': int(syn.drop(columns=['flow_key']).duplicated().sum()), 'constants_exact': f'{const_ok}/{len(constant)}'})
    X = load_syn_df(syn); p = fit_predict(X); g = f'CTGAN_k2000_s{s}'
    fid = fidelity(X)
    for m in MODELS: crow.append({**score(p[m], g, m), 'seed': s, **fid})
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\n', pd.DataFrame(logs).to_string(index=False))
cres = pd.DataFrame(crow); cres.to_csv(f'{KDIR}/ctgan_k2000_tstr_BA.csv', index=False)

# ---------- summary: mean ± std over seeds ----------
f1 = cres.pivot(index='seed', columns='Model', values='F1')[MODELS]
ba = cres.pivot(index='seed', columns='Model', values='BA')[MODELS]
f1['Mean'] = f1.mean(axis=1); ba['Mean'] = ba.mean(axis=1)
fid_s = cres.groupby('seed')[['KS', 'JSD', 'MMD2']].first()
ms = lambda d: (d.mean().round(3).astype(str) + ' ± ' + d.std().round(3).astype(str)).to_dict()
print('\nPer seed F1:\n', f1.round(3).to_string())
print('\nPer seed BA:\n', ba.round(3).to_string())
print('\nCTGAN k=2000  F1 :', ms(f1))
print('CTGAN k=2000  BA :', ms(ba))
print('CTGAN k=2000  fid:', ms(fid_s))
print('\nOct 13 paper  F1 : RF 0.910 ± 0.000 | LR 0.964 ± 0.032 | SVM 0.951 ± 0.047 | Mean 0.942')
print('Oct 13 paper  fid: KS 0.130 ± 0.010 | JSD 0.112 ± 0.007 | MMD2 0.056 ± 0.008')

# ---------- download everything ----------
shutil.make_archive('/content/ctgan_k2000', 'zip', KDIR)
from google.colab import files; files.download('/content/ctgan_k2000.zip')